# Introduction to Pandas
---

Pandas is considered another important Python library for scientific computing as it offers a multitude of built-in functionality to analyze and manipulate data. Its extensive documentation can be found [here](https://pandas.pydata.org/docs/). One of pandas' key objects is the **`DataFrame`**, a tabular data structure that efficiently handles numerical and time-series data (among others).

[This cheat sheet](https://github.com/pandas-dev/pandas/blob/master/doc/cheatsheet/Pandas_Cheat_Sheet.pdf) offers a good overview of what pandas and DataFrames are capable of.

## Overview
---

* [Loading Data](#Loading-Data)
* [Working with DataFrames](#Working-with-DataFrames)
* [Cleaning Data](#Cleaning-Data)
* [Saving Data](#Saving-Data)

### Loading Data

One great feature of pandas is its ability to load data into the notebook using only few lines of code.
Let us compare how to load data manually versus the *pandas way*.

First things first, we have to import the `numpy` and `pandas` libraries:

In [ ]:
import numpy as np
import pandas as pd # conda install pandas(?)

In [ ]:
# Load CSV file manually
manual_data = []
for line in open("./data/Important_Data.csv"):
    row = line.split(',')
    sample = list(map(float, row))
    manual_data.append(sample)

In [ ]:
# Look at the first few entries of the list
manual_data[0:10]

In [ ]:
# Convert list into a NumPy array
X = np.array(manual_data)
type(X)

In [ ]:
# Load CSV file using pandas
pandas_data = pd.read_csv("./data/Important_Data.csv", header=None)

In [ ]:
# Look at the first few entries of the DataFrame
pandas_data.head()

In [ ]:
# Convert DataFrame into NumPy array
Y = pandas_data.values
type(Y)

Another neat possibility to read data directly into a `DataFrame` is to query a database using SQL:

In [ ]:
# Create a connection to a SQLite database
import sqlite3
conn = sqlite3.connect("./data/Fashion_Sales.db")

In [ ]:
# Use SQL to select needed columns from the database
query = "SELECT * FROM Sales;"
sales_df = pd.read_sql(query, con=conn, index_col='ID', parse_dates=True)
conn.close()
sales_df.head()

Apart from the examples shown here, pandas offers many more functions for reading and writing from/to various formats. Please refer to the [documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html) for more information concerning IO operations. You could even retireve the data from the web:

In [ ]:
import openpyxl
sales_df2 = pd.read_excel(
   "https://raw.githubusercontent.com/datagy/mediumdata/refs/heads/master/sample_pivot.xlsx",
   parse_dates=True, engine="openpyxl"
)
sales_df2.head()

### Working with `DataFrames`

The following snippets of code should help you get an idea of what you can do with `DataFrames` in pandas. Please be aware that the complete scope of functionality reaches far beyond those examples. The official documentation contains a [10 minutes to pandas tutorial](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html), which will assist you to dive deeper into the topic.

In [ ]:
# Have a look at the last entries of a DataFrame
sales_df.tail()

In [ ]:
# Get some meta information on your DataFrame, including sample numbers and data types
sales_df.info()

In [ ]:
# Retrieve some statistical information from the data in your DataFrame
sales_df.describe()

In [ ]:
# Look up the dimensions of your DataFrame
sales_df.shape

### Selecting Data

In [ ]:
# A column, which results in a Series object
print (type(sales_df['Region']))
sales_df['Region']

In [ ]:
# or select two columns:
sales_df[['Region','Units']].head()


In [ ]:
# A row (2nd one). This uses our index ID, which is [0...999] here.
sales_df[2:3]

In [ ]:
# Better in this case to use the integer location with iloc (notation like in NumPy):
sales_df.iloc[2,:]

In [ ]:
# Perform a simple query, find all rows of the region East.
sales_df[sales_df.Region=='East']

In [ ]:
# # Find all rows of the region East, but only show the Region column.
sales_df['Region'][sales_df.Region=='East']

In [ ]:
# Finally, select region 'East' AND Units>10. Be carefull, because () is required!
sales_df[(sales_df.Region=='East') & (sales_df.Units>10)]

### Pivot Tables

`DataFrames` also offer the possibility to create and utilize [pivot tables](https://en.wikipedia.org/wiki/Pivot_table), a popular analysis component in business reporting. Since you are all familiar with SQL, this would correspond to the following GROUP BY: `SELECT Region, AVG(Sales) FROM sales GROUP BY Region;`

In [ ]:
# Create a simple pivot table that aggregates sales by region (aggregation defaults to 'mean')
sales_by_region = pd.pivot_table(sales_df, index='Region', values='Sales')
sales_by_region

In [ ]:
# Create another pivot table that aggregates sales by region, but sums up total sales
sales_by_region = pd.pivot_table(sales_df, index='Region', values='Sales', aggfunc='sum')
sales_by_region

In [ ]:
# Apply a filter to the pivot table that only lists 
# regions in which total sales are greater than 100000
sales_by_region[sales_by_region.Sales > 100000]

In [ ]:
# Apply a filter to the pivot table that only lists 
# regions in which total sales are greater than 100000 or less than 60000
sales_by_region[(sales_by_region.Sales > 100000) | (sales_by_region['Sales'] < 60000)]

In [ ]:
# Create a multi-index pivot table to add further groupings;
# Additionally, add another aggregation for sales counts
sales_by_region = pd.pivot_table(sales_df, index=['Region', 'Type'], values='Sales', aggfunc=['sum', len])
sales_by_region

In [ ]:
# Create a pivot table that aggregates sold units per clothing type
units_per_type = pd.pivot_table(sales_df, index='Type', values='Units', aggfunc='sum')
units_per_type

In [ ]:
# Add another dimension to see the sold units per clothing type by region
units_per_type = pd.pivot_table(sales_df, index='Type', values='Units', columns='Region', aggfunc='sum')
units_per_type

In [ ]:
# Create a simple plot to visualize the contents of your pivot table
units_per_type.plot(kind='bar')

As always, the [official documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html) is an excellent place to look for more detailed information regarding this topic.

Before we finish today's recap, let us apply some of what we learned to a task which is practically *always* necessary in real life data science tasks - data cleansing.

### Cleansing Data

As part of the **Data Preparation** phase, defined by [CRISP-DM](https://en.wikipedia.org/wiki/Cross-industry_standard_process_for_data_mining) (c.f. lecture slides), data almost always needs to be prepared before any model can be applied or an analysis task can be performed. One such preparation deals with cleansing messy data - and pandas provides some excellent tools to do so.

In the following example, we are looking at some dog registration information for New York City. The data is provided in a CSV file that contains 15 columns. First, we make sure to load only those columns that contain information useful to us:

In [ ]:
# Read specific columns from a CSV file
dogs_df = pd.read_csv("./data/NYC_Dogs.csv",
                      index_col='RowNumber',
                      usecols=['RowNumber', 'AnimalName', 'AnimalGender', 'AnimalBirthMonth', 'RegistrationFee'])
print(str(dogs_df.shape[0]) + ' rows')
dogs_df.head()


As you can see, the column headers are quite verbose. Let us change those names:

In [ ]:
# Change index name
dogs_df.index.name = 'id'
# Change column names
dogs_df.columns = ['name', 'gender', 'birth_month', 'registration_fee']
dogs_df

Next, we would like to change the way **`birth_month`** information is represented:

In [ ]:
# Have a look at the data types for each column
dogs_df.info()
print()
print(type(dogs_df['birth_month'].iloc[0]))
# birth_month does contain date/time information, but is currently of type object (str)

In [ ]:
# Change the data type for birth_month to datetime64
dogs_df.birth_month = pd.to_datetime(dogs_df.birth_month, format="%m/%d/%Y %I:%M:%S %p")
dogs_df.info()


In [ ]:
dogs_df

In [ ]:
# Format the date/time information to your style (the example shown here changes the data type back to object)
dogs_df.birth_month = dogs_df.birth_month.dt.strftime('%b-%Y')
dogs_df

One of the most important (and most typical) challenges when it comes to data cleaning is to handle samples that do not have an entry for every attribute. In most cases, missing data will lead to problems when performing analysis tasks. Therefore, it is necessary to find a strategy that takes care of those issues.

***Disclaimer***: The following examples should only represent some ideas of what you can do with missing data. However, it is not implied that these examples always portrait the best strategy for the given case.

In [ ]:
# See how many missing values there are in each column
dogs_df.isna().sum()
#dogs_df.isnull().sum()

When we start looking at the names column, there are 235 missing entries in total. However, we know from experience that there is an explicit entry for unknown dog names. It should be a good idea to change those missing values to that specific entry.

In [ ]:
# Have a look at the x most common entries for a column
dogs_df['name'].value_counts()[:5]

Wow, it seems that there are even two specific entries for unknown dog names. Let us fix this mess:

In [ ]:
# Replace empty entries in the name column with the string literal: 'UNKNOWN'
dogs_df.name =  dogs_df.name.fillna('UNKNOWN')
# Replace non-empty entries
dogs_df.name = dogs_df.name.replace('NAME NOT PROVIDED', 'UNKNOWN')
dogs_df['name'].value_counts()[:5]

The column **`registration_fee`** contains a large amount of missing values; far too many to remove them from the dataset altogether. One idea to handle this situation could be to choose a suitable default value for those entries:

In [ ]:
# Replace the empty entries in registration_fee with the mean value for that column
# (the mean value does not change after this operation)
print(dogs_df.registration_fee.mean())
dogs_df.registration_fee = dogs_df.registration_fee.fillna(dogs_df.registration_fee.mean())
print(dogs_df.registration_fee.mean())

**`gender`** as well as **`birth_month`** only have 1 entry missing each. It would be best to simply remove those samples entirely from our dataset:

In [ ]:
dogs_df.dropna(inplace=True)

**Note:** Clearly, the steps have to be done in excatly this sequence. Otherwise, we would have just thrown out all dogs without a proper entry in the `name`-column as well!

In [ ]:
# Get the total number of empty values in your DataFrame
dogs_df.isna().sum()

Eventually, we took care of all the missing values in our `DataFrame`. However, there still is something not quite right:

In [ ]:
# Take a look at the unique entries for the gender column
dogs_df.gender.unique()

In [ ]:
# Take a look how many samples there are for each unique entry
dogs_df.gender.value_counts()

As you can see, there are three unique values in the **`gender`** column. One of them being a non-empty string literal (in this case, a space character). Since there are typically three genders to assign, they are best set to 'D':

In [ ]:
# Replace gender from DataFrame based on attribute values
dogs_df.gender = dogs_df.gender.replace(' ', 'D')
dogs_df.gender.value_counts()

In [ ]:
dogs_df

Finally, we have a cleaned `DataFrame` that
* contains only those columns needed,
* has somewhat usable column headers,
* contains error-free data,
* does not contain missing values,
* has suitable data types.

Of course, there are many other things one might take into account when cleansing data (especially when you have business or domain knowledge). [This guide](https://towardsdatascience.com/the-ultimate-guide-to-data-cleaning-3969843991d4) offers a good overview of what else to look after when cleaning your data.

### Saving Data

Of course, one last important step is to save your results. Fortunately, pandas makes it as easy as reading data:

In [ ]:
# Saving DataFrame as CSV file
dogs_df.to_csv('./data/NYC_Dogs_Clean.csv')

Be reminded that there are many more ways to export and save data, described in the [documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html).